# LeetCode 383 — Ransom Note bằng Python

**Link đề:** https://leetcode.com/problems/ransom-note/
**Độ khó:** Easy · **Chủ đề:** Hash Table, String, Counting

```python
class Solution:
    def canConstruct(self, ransomNote: str, magazine: str) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu bài toán "**cắt chữ từ tạp chí để ghép thư**": mỗi chữ trong `magazine` chỉ dùng **một lần**.
2. Đi từ cách trực giác (tìm và xóa từng chữ) `O(n·m)` → **đếm tần suất** `O(n + m)`.
3. Nắm 3 công cụ đếm: `dict`, `collections.Counter`, và **mảng 26 phần tử** với `ord(ch) - ord('a')`.
4. Trace bộ đếm từng bước; chứng minh tính đúng; so sánh nhiều cách.
5. Nhận ra pattern "**đủ nguyên liệu?**" (multiset containment) trong các bài khác.

## 1. Đề bài

Cho hai chuỗi `ransomNote` (lá thư cần ghép) và `magazine` (tạp chí). Trả `True` nếu có thể ghép `ransomNote` bằng các chữ cái lấy từ `magazine`, ngược lại `False`.

**Mỗi chữ cái trong `magazine` chỉ được dùng một lần.**

| `ransomNote` | `magazine` | Kết quả | Vì sao |
|---|---|:---:|---|
| `"a"` | `"b"` | `False` | không có chữ `a` |
| `"aa"` | `"ab"` | `False` | cần 2 chữ `a`, tạp chí chỉ có 1 |
| `"aa"` | `"aab"` | `True` | đủ 2 chữ `a` (chữ `b` thừa không sao) |

**Constraints:** `1 <= len(ransomNote), len(magazine) <= 10^5`; chỉ gồm chữ cái tiếng Anh **thường**.

**Câu hỏi then chốt:** với **mỗi chữ cái** `c`, tạp chí có **ít nhất** bằng số lần `c` xuất hiện trong thư không? Thứ tự chữ **không quan trọng**.

In [1]:
# Ví dụ 2 bằng tay: đếm chữ a
note, mag = "aa", "ab"
print("Thư cần", note.count("a"), "chữ a; tạp chí có", mag.count("a"), "-> đủ?", mag.count("a") >= note.count("a"))

Thư cần 2 chữ a; tạp chí có 1 -> đủ? False


## 2. Nền tảng Python

### 2.1. Chuỗi (`str`) là bất biến; duyệt từng ký tự

`for ch in s:` lấy lần lượt từng ký tự. Chuỗi **không sửa được tại chỗ** (immutable): `s[0] = "x"` gây lỗi. Muốn "xóa" một ký tự phải tạo **chuỗi mới** — tốn `O(len(s))`.

In [2]:
s = "aab"
for ch in s:
    print(ch, end=" ")
print()
try:
    s[0] = "x"
except TypeError as e:
    print("Không sửa được chuỗi:", e)
print("Tạo chuỗi mới bỏ ký tự đầu:", s[1:], "| replace 1 lần:", s.replace("a", "", 1))

a a b 
Không sửa được chuỗi: 'str' object does not support item assignment
Tạo chuỗi mới bỏ ký tự đầu: ab | replace 1 lần: ab


`s.replace(old, new, count)` thay **tối đa `count`** lần xuất hiện của `old`. `s[1:]` là **slicing** — lấy từ chỉ số 1 đến hết.

### 2.2. `dict` đếm tần suất

Mẫu quen thuộc: `cnt[ch] = cnt.get(ch, 0) + 1`.

In [3]:
cnt = {}
for ch in "aab":
    cnt[ch] = cnt.get(ch, 0) + 1
print(cnt)
print("cnt.get('z', 0) =", cnt.get("z", 0))

{'a': 2, 'b': 1}
cnt.get('z', 0) = 0


### 2.3. `collections.Counter`

`Counter(s)` đếm sẵn; khóa thiếu trả `0`. Phép **trừ** `c1 - c2` giữ lại các khóa có hiệu **dương** (khóa nào ≤ 0 bị bỏ). Từ Python 3.10, `c1 <= c2` kiểm tra "mọi khóa của `c1` có số đếm ≤ ở `c2`".

In [4]:
from collections import Counter
need = Counter("aab")
have = Counter("abcab")
print("need =", need, "| have =", have)
print("need - have =", need - have, "(rỗng nghĩa là đủ)")
print("Counter('aaa') - have =", Counter("aaa") - have, "(thiếu 1 chữ a)")
print("need <= have:", need <= have)

need = Counter({'a': 2, 'b': 1}) | have = Counter({'a': 2, 'b': 2, 'c': 1})
need - have = Counter() (rỗng nghĩa là đủ)
Counter('aaa') - have = Counter({'a': 1}) (thiếu 1 chữ a)
need <= have: True


### 2.4. `ord()` và mảng 26 phần tử

`ord(ch)` trả mã số Unicode của ký tự: `ord('a') = 97`, `ord('b') = 98`, ... `ord('z') = 122`. Vì vậy `ord(ch) - ord('a')` ánh xạ `'a'..'z'` thành chỉ số `0..25` — dùng làm vị trí trong một list 26 số đếm.

In [5]:
for ch in "abz":
    print(ch, "-> ord =", ord(ch), "-> chỉ số =", ord(ch) - ord("a"))
counts = [0] * 26
for ch in "aab":
    counts[ord(ch) - ord("a")] += 1
print(counts[:3], "... (chỉ in 3 ô đầu: a, b, c)")

a -> ord = 97 -> chỉ số = 0
b -> ord = 98 -> chỉ số = 1
z -> ord = 122 -> chỉ số = 25
[2, 1, 0] ... (chỉ in 3 ô đầu: a, b, c)


## 3. Cách 1 — Trực giác: "cắt" từng chữ ra khỏi tạp chí

Với mỗi chữ `ch` của thư: tìm `ch` trong tạp chí; có thì **xóa** một bản (vì đã dùng), không có thì `False`.

In [6]:
def can_construct_remove(ransomNote: str, magazine: str) -> bool:
    for ch in ransomNote:
        idx = magazine.find(ch)               # vị trí đầu tiên, -1 nếu không có
        if idx == -1:
            return False
        magazine = magazine[:idx] + magazine[idx + 1:]   # tạo chuỗi mới bỏ ký tự idx
    return True

for note, mag in [("a", "b"), ("aa", "ab"), ("aa", "aab")]:
    print(note, mag, "->", can_construct_remove(note, mag))

a b -> False
aa ab -> False
aa aab -> True


`s.find(x)` trả chỉ số **đầu tiên** của `x` trong `s`, hoặc `-1` nếu không có (khác `s.index(x)` ném lỗi). `magazine[:idx] + magazine[idx + 1:]` ghép phần trước và sau vị trí `idx` → chuỗi mới không có ký tự đó.

### 3.1. Nút thắt

- Mỗi lần `find` và mỗi lần ghép chuỗi tốn `O(m)` → tổng `O(n·m)`. Với `n = m = 10^5` là `10^10` thao tác → quá chậm.
- Ta không cần biết **vị trí** chữ trong tạp chí; chỉ cần biết **còn bao nhiêu** bản của mỗi chữ.

**Insight:** đếm số lần mỗi chữ xuất hiện trong tạp chí **một lần** (`O(m)`), rồi "tiêu" dần bộ đếm khi đọc thư (`O(1)` mỗi chữ).

## 4. Cách 2 — Đếm tạp chí rồi tiêu dần (lời giải chính)

**Đoạn 1 — đếm nguồn cung:** `cnt[c]` = số chữ `c` còn trong tạp chí.

**Đoạn 2 — tiêu thụ:** duyệt từng `ch` của thư; nếu `cnt.get(ch, 0) == 0` → hết chữ → `False`.

**Đoạn 3 — trừ:** ngược lại `cnt[ch] -= 1`.

Đi hết thư mà không thiếu → `True`.

In [7]:
def can_construct_dict(ransomNote: str, magazine: str) -> bool:
    cnt = {}
    for c in magazine:                     # Đoạn 1
        cnt[c] = cnt.get(c, 0) + 1
    for ch in ransomNote:
        if cnt.get(ch, 0) == 0:            # Đoạn 2
            return False
        cnt[ch] -= 1                       # Đoạn 3
    return True

for note, mag in [("a", "b"), ("aa", "ab"), ("aa", "aab")]:
    print(note, mag, "->", can_construct_dict(note, mag))

a b -> False
aa ab -> False
aa aab -> True


### 4.1. Tối ưu nhỏ: thư dài hơn tạp chí thì chắc chắn `False`

Nếu `len(ransomNote) > len(magazine)` thì không thể đủ chữ → trả `False` ngay, khỏi đếm.

In [8]:
print(len("aaaa") > len("aab"), "-> trả False ngay cho ('aaaa', 'aab')")

True -> trả False ngay cho ('aaaa', 'aab')


## 5. TRACE từng bước — một ca đủ chữ và một ca thiếu chữ

Sau Đoạn 1 với `magazine = "baa"`: `cnt = {b: 1, a: 2}`.

| Bước | `ch` | `cnt[ch]` trước | Đủ? | `cnt` sau |
|---:|:---:|---:|:---:|---|
| 1 | a | 2 | ✔ | `{b: 1, a: 1}` |
| 2 | a | 1 | ✔ | `{b: 1, a: 0}` |
| 3 | b | 1 | ✔ | `{b: 0, a: 0}` |

→ `True`.

Với `ransomNote = "aab"`, `magazine = "ab"`: `cnt = {a: 1, b: 1}`; bước 1 dùng `a` → `{a: 0, b: 1}`; bước 2 cần `a` nhưng `cnt[a] = 0` → **`False`** ngay, không cần xét tiếp.

In [9]:
def trace_can_construct(ransomNote, magazine):
    cnt = {}
    for c in magazine:
        cnt[c] = cnt.get(c, 0) + 1
    print(f"note={ransomNote!r}, magazine={magazine!r} -> cnt ban đầu = {cnt}")
    for step, ch in enumerate(ransomNote, start=1):
        before = cnt.get(ch, 0)
        if before == 0:
            print(f"Bước {step}: cần '{ch}', còn {before} -> THIẾU -> False")
            return False
        cnt[ch] -= 1
        print(f"Bước {step}: cần '{ch}', còn {before} -> dùng 1 -> cnt = {cnt}")
    print("Đủ chữ cho cả lá thư -> True")
    return True

_ = trace_can_construct("aab", "baa")
print()
_ = trace_can_construct("aab", "ab")

note='aab', magazine='baa' -> cnt ban đầu = {'b': 1, 'a': 2}
Bước 1: cần 'a', còn 2 -> dùng 1 -> cnt = {'b': 1, 'a': 1}
Bước 2: cần 'a', còn 1 -> dùng 1 -> cnt = {'b': 1, 'a': 0}
Bước 3: cần 'b', còn 1 -> dùng 1 -> cnt = {'b': 0, 'a': 0}
Đủ chữ cho cả lá thư -> True

note='aab', magazine='ab' -> cnt ban đầu = {'a': 1, 'b': 1}
Bước 1: cần 'a', còn 1 -> dùng 1 -> cnt = {'a': 0, 'b': 1}
Bước 2: cần 'a', còn 0 -> THIẾU -> False


## 6. Cách 3 — `Counter` một dòng

"Thư cần" trừ "tạp chí có": nếu còn dư chữ nào (hiệu dương) thì thiếu.

```python
return not (Counter(ransomNote) - Counter(magazine))
```

`not counter_rỗng` là `True` vì một `Counter`/`dict` rỗng được coi là **falsy** (giống `False` trong `if`).

In [10]:
def can_construct_counter(ransomNote: str, magazine: str) -> bool:
    return not (Counter(ransomNote) - Counter(magazine))

def can_construct_counter_le(ransomNote: str, magazine: str) -> bool:
    return Counter(ransomNote) <= Counter(magazine)      # Python 3.10+

print(bool(Counter()), bool(Counter("a")))
for note, mag in [("a", "b"), ("aa", "ab"), ("aa", "aab")]:
    print(note, mag, "->", can_construct_counter(note, mag), can_construct_counter_le(note, mag))

False True
a b -> False False
aa ab -> False False
aa aab -> True True


## 7. Cách 4 — Mảng 26 phần tử (nhanh, bộ nhớ cố định)

Vì chỉ có 26 chữ thường, thay `dict` bằng list `counts` dài 26. Truy cập theo chỉ số luôn `O(1)` (không cần băm).

In [11]:
def can_construct_array(ransomNote: str, magazine: str) -> bool:
    if len(ransomNote) > len(magazine):
        return False
    counts = [0] * 26
    for c in magazine:
        counts[ord(c) - ord("a")] += 1
    for ch in ransomNote:
        i = ord(ch) - ord("a")
        if counts[i] == 0:
            return False
        counts[i] -= 1
    return True

for note, mag in [("a", "b"), ("aa", "ab"), ("aa", "aab")]:
    print(note, mag, "->", can_construct_array(note, mag))

a b -> False
aa ab -> False
aa aab -> True


## 8. Cách 5 — Đếm theo từng chữ cái bằng `str.count`

Chỉ có 26 chữ cái khác nhau, nên với **mỗi chữ khác nhau** trong thư, so `ransomNote.count(c) <= magazine.count(c)`. Mỗi lần `count` quét chuỗi `O(n)` hoặc `O(m)`, tối đa 26 chữ → `O(26·(n + m))` = `O(n + m)`. Cách này ngắn và thực tế chạy nhanh vì `count` viết bằng C.

`all(iterable)` trả `True` nếu **mọi** phần tử đều đúng (và dừng sớm khi gặp phần tử sai).

In [12]:
def can_construct_count(ransomNote: str, magazine: str) -> bool:
    return all(ransomNote.count(c) <= magazine.count(c) for c in set(ransomNote))

print(all([True, True]), all([True, False]), all([]))
for note, mag in [("a", "b"), ("aa", "ab"), ("aa", "aab")]:
    print(note, mag, "->", can_construct_count(note, mag))

True False True
a b -> False
aa ab -> False
aa aab -> True


`(... for c in set(ransomNote))` là **generator expression** — giống list comprehension nhưng không tạo list, sinh từng giá trị khi `all` cần. `set(ransomNote)` để mỗi chữ chỉ kiểm tra một lần.

## 9. Code nộp LeetCode

In [13]:
class Solution:
    def canConstruct(self, ransomNote: str, magazine: str) -> bool:
        if len(ransomNote) > len(magazine):
            return False
        cnt = {}
        for c in magazine:
            cnt[c] = cnt.get(c, 0) + 1
        for ch in ransomNote:
            if cnt.get(ch, 0) == 0:
                return False
            cnt[ch] -= 1
        return True


sol = Solution()
print(sol.canConstruct("a", "b"), sol.canConstruct("aa", "ab"), sol.canConstruct("aa", "aab"))

False False True


Bản nộp ngắn:

```python
from collections import Counter

class Solution:
    def canConstruct(self, ransomNote: str, magazine: str) -> bool:
        return not (Counter(ransomNote) - Counter(magazine))
```

## 10. Vì sao đúng?

Gọi `need(c)` = số lần `c` trong thư, `have(c)` = số lần trong tạp chí. Ghép được ⇔ **với mọi `c`: `need(c) ≤ have(c)`** (mỗi chữ của thư cần một chữ riêng của tạp chí, thứ tự không quan trọng).

**Invariant (Cách 2):** *sau khi xử lý `k` ký tự đầu của thư, `cnt[c] = have(c) − (số lần c trong k ký tự đó)` và mọi giá trị `cnt[c] ≥ 0`.*

- Nếu gặp `ch` mà `cnt[ch] = 0` → số `ch` cần (tính cả ký tự này) đã vượt `have(ch)` → `need(ch) > have(ch)` → `False` đúng.
- Nếu đi hết thư không bị chặn → mọi `cnt[c] ≥ 0` ⇒ `need(c) ≤ have(c)` với mọi `c` → `True` đúng.

## 11. Độ phức tạp

Gọi `n = len(ransomNote)`, `m = len(magazine)`, `Σ = 26` (kích thước bảng chữ cái).

| Cách | Time | Space |
|---|---|---|
| 1. Tìm & xóa | `O(n·m)` | `O(m)` (chuỗi mới) |
| 2. Dict đếm | `O(n + m)` trung bình | `O(Σ)` |
| 3. `Counter` trừ | `O(n + m)` | `O(Σ)` |
| 4. Mảng 26 | `O(n + m)` | `O(Σ)` = `O(1)` |
| 5. `str.count` theo chữ | `O(Σ·(n + m))` | `O(Σ)` |

Vì `Σ` cố định, bộ nhớ của các cách 2–5 được xem là `O(1)`.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [14]:
methods = {
    "remove": can_construct_remove,
    "dict": can_construct_dict,
    "counter": can_construct_counter,
    "counter_le": can_construct_counter_le,
    "array": can_construct_array,
    "count": can_construct_count,
    "Solution": Solution().canConstruct,
}
cases = [
    ("a", "b", False), ("aa", "ab", False), ("aa", "aab", True),       # mẫu đề
    ("a", "a", True),                                                   # 1 ký tự
    ("abc", "cba", True),                                               # đảo thứ tự vẫn được
    ("aab", "baa", True),
    ("aab", "ab", False),                                               # thư dài hơn
    ("z", "abcdefghijklmnopqrstuvwxy", False),                          # thiếu đúng chữ z
    ("bg", "efjbdfbdgfjhhaiigfhbaejahgfbbgbjagbddfgdiaigdadhcfcj", True),
]
for note, mag, exp in cases:
    for name, f in methods.items():
        assert f(note, mag) == exp, (name, note, mag)
    print(f"{note!r:>6} / {mag[:20]!r:<24} -> {exp} PASS")

   'a' / 'b'                      -> False PASS
  'aa' / 'ab'                     -> False PASS
  'aa' / 'aab'                    -> True PASS
   'a' / 'a'                      -> True PASS
 'abc' / 'cba'                    -> True PASS
 'aab' / 'baa'                    -> True PASS
 'aab' / 'ab'                     -> False PASS
   'z' / 'abcdefghijklmnopqrst'   -> False PASS
  'bg' / 'efjbdfbdgfjhhaiigfhb'   -> True PASS


In [15]:
import random, string
rng = random.Random(383)
for _ in range(1000):
    note = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 8)))
    mag = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 10)))
    exp = can_construct_remove(note, mag)          # oracle chậm nhưng dễ tin
    for name, f in methods.items():
        assert f(note, mag) == exp, (name, note, mag)
big_mag = "".join(rng.choice(string.ascii_lowercase) for _ in range(10**5))
assert Solution().canConstruct(big_mag[:50000], big_mag) is True
print("1000 test ngẫu nhiên + 1 test 10^5 ký tự PASS")

1000 test ngẫu nhiên + 1 test 10^5 ký tự PASS


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Chỉ kiểm tra "có mặt" mà quên số lần
`all(ch in magazine for ch in ransomNote)` → `("aa", "ab")` trả `True` sai (chữ `a` bị dùng 2 lần).

### Lỗi 2 — Đếm nhầm chiều (đếm thư, tiêu tạp chí) rồi điều kiện ngược
Đếm `ransomNote` rồi trừ theo `magazine` cũng làm được, nhưng điều kiện cuối phải là "**mọi số đếm ≤ 0**". Viết nhầm thành "có số nào bằng 0" sẽ sai.

### Lỗi 3 — Dùng `set` để so sánh
`set(ransomNote) <= set(magazine)` bỏ mất số lần → giống lỗi 1.

### Lỗi 4 — `cnt[ch]` thay vì `cnt.get(ch, 0)` với dict thường
Chữ không có trong tạp chí → `KeyError`.

In [16]:
def wrong_presence(note, mag):
    return all(ch in mag for ch in note)

def wrong_set(note, mag):
    return set(note) <= set(mag)

def wrong_keyerror(note, mag):
    cnt = {}
    for c in mag:
        cnt[c] = cnt.get(c, 0) + 1
    for ch in note:
        if cnt[ch] == 0:
            return False
        cnt[ch] -= 1
    return True

print("Lỗi 1:", wrong_presence("aa", "ab"), "(đúng: False)")
print("Lỗi 3:", wrong_set("aa", "ab"), "(đúng: False)")
try:
    wrong_keyerror("a", "b")
except KeyError as e:
    print("Lỗi 4: KeyError", e)

Lỗi 1: True (đúng: False)
Lỗi 3: True (đúng: False)
Lỗi 4: KeyError 'a'


## 14. Mẹo nhận dạng pattern

- "Dùng chữ/vật từ A để tạo B, mỗi thứ dùng **một lần**" → so sánh **bộ đếm**: `need ≤ have` cho mọi khóa.
- Bảng chữ cái nhỏ, cố định (26 chữ thường) → mảng 26 thay cho dict.
- "Thứ tự không quan trọng" là tín hiệu mạnh cho đếm tần suất (không cần sắp xếp/hai con trỏ).
- Bài anh em: **242 Valid Anagram** (`need == have`), **350 Intersection II**, **1160 Find Words That Can Be Formed by Characters**, **389 Find the Difference**, **2287 Rearrange Characters to Make Target String** (đếm số **bản** ghép được).

## 15. Biến thể

### 15.1. Ghép được tối đa bao nhiêu bản của `target`?

Số bản = `min(have(c) // need(c))` trên mọi `c` trong `target` (ý tưởng LeetCode 2287 / 1189 "Maximum Number of Balloons").

In [17]:
def max_copies(target: str, source: str) -> int:
    need, have = Counter(target), Counter(source)
    return min(have[c] // need[c] for c in need)

print(max_copies("balloon", "loonbalxballpoon"))   # 2
print(max_copies("aa", "aaaaa"))                     # 2

2
2


### 15.2. Thiếu những chữ nào?

Trả về các chữ còn thiếu (kèm số lượng) để ghép được thư: chính là `Counter(note) - Counter(magazine)`.

In [18]:
def missing_letters(note: str, magazine: str) -> dict:
    return dict(Counter(note) - Counter(magazine))

print(missing_letters("hello", "hlo"))     # thiếu 1 'e' và 1 'l'

{'e': 1, 'l': 1}


## 16. Bài tập tự luyện (TODO)

**Bài 1 (dự đoán output).** `Counter("abca") - Counter("aab")` in ra gì? Kết luận `canConstruct("abca", "aab")`?

**Bài 2 (tính tay).** Lập bảng trace Cách 2 cho `ransomNote = "bba"`, `magazine = "abab"`.

**Bài 3 (điền chỗ trống).** Hoàn thiện `exercise_array` (mảng 26).

**Bài 4 (viết lại).** Viết `exercise_reverse_count`: đếm **thư** trước, rồi duyệt **tạp chí** để giảm; cuối cùng kiểm tra mọi số đếm `<= 0`.

**Bài 5 (biến thể).** `can_form_all(words, chars)`: trả tổng độ dài các từ trong `words` ghép được từ `chars` (mỗi từ dùng lại toàn bộ `chars`, LeetCode 1160).

**Bài 6 (mini challenge).** Thư và tạp chí gồm **mọi ký tự Unicode** (có hoa, số, dấu cách, tiếng Việt). Viết `can_construct_unicode` (mảng 26 không còn dùng được — vì sao?).

In [19]:
def exercise_array(ransomNote: str, magazine: str) -> bool:
    # TODO 1: counts = [0] * 26
    # TODO 2: đếm magazine bằng ord(c) - ord('a')
    # TODO 3: duyệt ransomNote, thiếu thì False, đủ thì trừ
    pass

# assert exercise_array("aa", "aab") and not exercise_array("aa", "ab")

In [20]:
def exercise_reverse_count(ransomNote: str, magazine: str) -> bool:
    # TODO
    pass

# assert exercise_reverse_count("aa", "aab") and not exercise_reverse_count("aa", "ab")

In [21]:
def can_form_all(words: list[str], chars: str) -> int:
    # TODO
    pass

# assert can_form_all(["cat", "bt", "hat", "tree"], "atach") == 6

In [22]:
def can_construct_unicode(note: str, magazine: str) -> bool:
    # TODO
    pass

# assert can_construct_unicode("Xin chào!", "!oàhc niX ") and not can_construct_unicode("ĐĐ", "Đd")

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | trừ giữ hiệu dương | a:2−2, b:1−1, c:1−0 | còn `{'c': 1}` |
| 2 | đếm `abab` trước | `{a: 2, b: 2}` | b → b → a, đều đủ |
| 3 | giống Cách 4 | chỉ số `ord(c) - 97` | `if counts[i] == 0: return False` |
| 4 | `need` dương = còn thiếu | giảm khi gặp chữ ở tạp chí | `all(v <= 0 for v in cnt.values())` |
| 5 | mỗi từ là một bài Ransom Note | dùng lại `Counter(chars)` | cộng `len(w)` nếu ghép được |
| 6 | không chắc ký tự thuộc `a..z` | dùng `dict`/`Counter` | `not (Counter(note) - Counter(magazine))` |

## 18. Đáp án tham khảo + Template + Checklist

**Bài 1:** `Counter({'c': 1})` → còn thiếu `c` → `False`.

**Bài 2:** `cnt = {a: 2, b: 2}` → `b`: còn 2 → `{a:2, b:1}` → `b`: còn 1 → `{a:2, b:0}` → `a`: còn 2 → `{a:1, b:0}` → `True`.

**Bài 6:** chữ hoa, dấu, chữ có dấu tiếng Việt có `ord` ngoài `97..122` → chỉ số vượt `0..25`; dùng `Counter`.

In [23]:
def answer_array(ransomNote, magazine):
    counts = [0] * 26
    for c in magazine:
        counts[ord(c) - ord("a")] += 1
    for ch in ransomNote:
        i = ord(ch) - ord("a")
        if counts[i] == 0:
            return False
        counts[i] -= 1
    return True

def answer_reverse_count(ransomNote, magazine):
    cnt = Counter(ransomNote)
    for c in magazine:
        if c in cnt:
            cnt[c] -= 1
    return all(v <= 0 for v in cnt.values())

def answer_can_form_all(words, chars):
    have = Counter(chars)
    return sum(len(w) for w in words if not (Counter(w) - have))

def answer_can_construct_unicode(note, magazine):
    return not (Counter(note) - Counter(magazine))

print("Bài 1:", Counter("abca") - Counter("aab"))
_ = trace_can_construct("bba", "abab")
assert answer_array("aa", "aab") and not answer_array("aa", "ab")
assert answer_reverse_count("aa", "aab") and not answer_reverse_count("aa", "ab")
assert answer_can_form_all(["cat", "bt", "hat", "tree"], "atach") == 6
assert answer_can_construct_unicode("Xin chào!", "!oàhc niX ") and not answer_can_construct_unicode("ĐĐ", "Đd")
print("Đáp án PASS toàn bộ!")

Bài 1: Counter({'c': 1})
note='bba', magazine='abab' -> cnt ban đầu = {'a': 2, 'b': 2}
Bước 1: cần 'b', còn 2 -> dùng 1 -> cnt = {'a': 2, 'b': 1}
Bước 2: cần 'b', còn 1 -> dùng 1 -> cnt = {'a': 2, 'b': 0}
Bước 3: cần 'a', còn 2 -> dùng 1 -> cnt = {'a': 1, 'b': 0}
Đủ chữ cho cả lá thư -> True
Đáp án PASS toàn bộ!


### Template nhớ nhanh

```python
class Solution:
    def canConstruct(self, ransomNote: str, magazine: str) -> bool:
        cnt = {}
        for c in magazine:                 # 1) đếm nguồn cung
            cnt[c] = cnt.get(c, 0) + 1
        for ch in ransomNote:              # 2) tiêu thụ
            if cnt.get(ch, 0) == 0:
                return False               #    thiếu -> dừng sớm
            cnt[ch] -= 1
        return True
```

### Checklist submit
- [ ] Đếm **số lần**, không chỉ kiểm tra "có mặt"?
- [ ] Trừ bộ đếm sau mỗi lần dùng một chữ?
- [ ] Không `KeyError` (dùng `.get` hoặc `Counter`)?
- [ ] Đã thử: thiếu chữ, đủ chữ nhưng thiếu số lần, thư dài hơn tạp chí, đảo thứ tự?
- [ ] Giải thích được vì sao bộ nhớ là `O(1)` (26 chữ cái)?